<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/03_train_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive[cite: 4]
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn[cite: 4]

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab[cite: 4]

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive[cite: 4]
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)[cite: 3]
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)[cite: 3, 4]
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại[cite: 3]
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device)                 # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)
print('Thư mục dữ liệu processed tồn tại:', os.path.exists(PROCESSED_DATA_DIR)) # Kiểm tra thư mục dữ liệu đã sẵn sàng chưa[cite: 3]

# ===== THÊM ĐOẠN NÀY VÀO CELL 1 ĐỂ CỐ ĐỊNH KẾT QUẢ =====
#import random
#import numpy as np
#import torch

#def set_seed(seed=42):
#    random.seed(seed)
#    np.random.seed(seed)
#    torch.manual_seed(seed)
#    torch.cuda.manual_seed(seed)
#    torch.cuda.manual_seed_all(seed)
#    torch.backends.cudnn.deterministic = True
#    torch.backends.cudnn.benchmark = False

#set_seed(42)  # Cố định hạt giống ngẫu nhiên là 42

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda
Thư mục dữ liệu processed tồn tại: True


In [2]:
# ===== CELL 2: ĐỊNH NGHĨA PYTORCH DATASET (TỐI ƯU TỐC ĐỘ) =====
from torch.utils.data import Dataset, DataLoader  # Import công cụ quản lý và nạp dữ liệu theo Batch
import numpy as np                                # Import NumPy để đọc file .npy

class PMSMDataset(Dataset):
    """
    Lớp Dataset tùy chỉnh nạp Input (X) và Target (Y) cho mạng MLP thuần dữ liệu.
    """
    def __init__(self, data_dir, split='train'):
        # Nạp mảng Input và Target từ ổ đĩa và chuyển trực tiếp sang Tensor float32 của PyTorch
        self.X = torch.tensor(np.load(os.path.join(data_dir, f'X_{split}.npy')), dtype=torch.float32)
        self.Y = torch.tensor(np.load(os.path.join(data_dir, f'Y_{split}.npy')), dtype=torch.float32)

    def __len__(self):
        # Trả về tổng số lượng mẫu trong tập dữ liệu
        return len(self.X)

    def __getitem__(self, idx):
        # Lấy ra cặp dữ liệu (Input, Target) tại vị trí chỉ số idx
        return self.X[idx], self.Y[idx]

# Khởi tạo 3 đối tượng Dataset cho Train, Validation và Test
train_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='train')
val_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='validation')
test_dataset = PMSMDataset(PROCESSED_DATA_DIR, split='test')

# Khởi tạo DataLoader: tăng batch_size lên 512/1024 và bật pin_memory=True để truyền dữ liệu từ RAM sang GPU nhanh nhất
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, pin_memory=True)

print(f'Số mẫu Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}')

Số mẫu Train: 900000 | Validation: 300000 | Test: 300000


In [3]:
# ===== CELL 3: KIẾN TRÚC MẠNG MLP =====
import torch.nn as nn # Import module chứa các lớp mạng Neural của PyTorch

class PMSM_MLP(nn.Module):
    """
    Kiến trúc Mạng Multi-Layer Perceptron (MLP) thuần dữ liệu (Data-driven).
    Nhận 5 nút đầu vào [ud, uq, Id, Iq, nm] và dự đoán 3 nút đầu ra [Rs, Ld, Lq].[cite: 2, 3]
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_MLP, self).__init__()

        # Xây dựng các lớp liên kết đầy đủ (Fully Connected / Dense Layers)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # Lớp đầu vào: 5 nút -> 128 nút ẩn
            nn.Tanh(),                          # Hàm kích hoạt Tanh tạo thuộc tính phi tuyến
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 1: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 2: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, output_dim)   # Lớp đầu ra: 128 -> 3 tham số ước lượng [Rs, Ld, Lq][cite: 2, 3]
        )

    def forward(self, x):
        # Hàm lan truyền tiến (Forward Pass) truyền tensor x qua các lớp mạng
        return self.network(x)

# Khởi tạo mô hình MLP và chuyển toàn bộ thông số lên GPU (cuda)
model = PMSM_MLP(input_dim=5, output_dim=3, hidden_dim=128).to(device)
print(model) # In cấu trúc mạng MLP ra màn hình

PMSM_MLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [4]:
# ===== CELL 4: HUẤN LUYỆN MÔ HÌNH MLP (TỐI ƯU TỐC ĐỘ & LƯU LOCAL) =====
import torch.optim as optim  # Import bộ tối ưu hóa trọng số Optimizer
import shutil                # Import thư viện shutil để copy file từ Colab sang Google Drive
import time                  # Import thư viện time để đo thời gian huấn luyện

criterion = nn.MSELoss()                             # Hàm mất mát sai số bình phương trung bình (MSE)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Bộ tối ưu Adam với tốc độ học khởi tạo lr = 0.001
# Bộ điều chỉnh tốc độ học tự động: giảm lr đi một nửa nếu Val Loss không cải thiện sau 10 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

epochs = 100                 # Tổng số vòng lặp huấn luyện
best_val_loss = float('inf') # Biến lưu kỷ lục Val Loss thấp nhất

# Lưu file trọng số tạm thời trên ổ cứng cục bộ của Colab (/content) để tránh nghẽn mạng khi ghi liên tục lên Drive
LOCAL_CKPT = '/content/best_mlp_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN MLP ===")
start_time = time.time()  # Ghi lại mốc thời gian bắt đầu chạy

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model.train()  # Bật chế độ huấn luyện
    train_loss = 0.0

    for inputs, targets in train_loader:
        # Chuyển dữ liệu Batch lên GPU (non_blocking=True giúp truyền bất đồng bộ nhanh hơn)
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()          # Xóa gradient của bước lặp trước
        outputs = model(inputs)        # Lan truyền tiến: dự đoán tham số [Rs, Ld, Lq]
        loss = criterion(outputs, targets) # Tính sai số MSE giữa dự đoán và nhãn thực tế
        loss.backward()                # Lan truyền ngược: tính đạo hàm theo các trọng số
        optimizer.step()               # Cập nhật trọng số mạng

        train_loss += loss.item() * inputs.size(0) # Cộng dồn sai số của Batch

    train_loss /= len(train_dataset) # Tính sai số trung bình trên toàn tập Train

    # --- GIAI ĐOẠN 2: ĐÁNH GIÁ (VALIDATION) ---
    model.eval()  # Bật chế độ đánh giá
    val_loss = 0.0
    with torch.no_grad():  # Tắt tính toán gradient để tiết kiệm VRAM và tăng tốc
        for inputs, targets in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            val_loss += criterion(outputs, targets).item() * inputs.size(0)

    val_loss /= len(val_dataset) # Tính sai số trung bình trên tập Validation
    scheduler.step(val_loss)     # Cập nhật bộ điều chỉnh Learning Rate theo Val Loss

    # Lưu mô hình tốt nhất vào ổ cứng cục bộ /content
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    # In thông số huấn luyện và thời gian trôi qua sau mỗi 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        elapsed = (time.time() - start_time) / 60.0
        print(f"Epoch [{epoch:3d}/{epochs}] | Train Loss (MSE): {train_loss:.6f} | Val Loss (MSE): {val_loss:.6f} | Thời gian: {elapsed:.2f} phút")

# Sau khi huấn luyện xong 100 epochs, copy bản trọng số tốt nhất sang Google Drive 1 lần duy nhất
shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Đã hoàn thành train trong {(time.time() - start_time)/60:.2f} phút! Lưu tại: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN MLP ===
Epoch [  1/100] | Train Loss (MSE): 0.257509 | Val Loss (MSE): 0.657604 | Thời gian: 0.16 phút
Epoch [ 10/100] | Train Loss (MSE): 0.093644 | Val Loss (MSE): 0.580939 | Thời gian: 1.58 phút
Epoch [ 20/100] | Train Loss (MSE): 0.066336 | Val Loss (MSE): 0.632466 | Thời gian: 3.17 phút
Epoch [ 30/100] | Train Loss (MSE): 0.061502 | Val Loss (MSE): 0.642171 | Thời gian: 4.74 phút
Epoch [ 40/100] | Train Loss (MSE): 0.057266 | Val Loss (MSE): 0.577613 | Thời gian: 6.33 phút
Epoch [ 50/100] | Train Loss (MSE): 0.054279 | Val Loss (MSE): 0.538629 | Thời gian: 7.91 phút
Epoch [ 60/100] | Train Loss (MSE): 0.053107 | Val Loss (MSE): 0.498723 | Thời gian: 9.49 phút
Epoch [ 70/100] | Train Loss (MSE): 0.051833 | Val Loss (MSE): 0.519830 | Thời gian: 11.06 phút
Epoch [ 80/100] | Train Loss (MSE): 0.050657 | Val Loss (MSE): 0.507089 | Thời gian: 12.65 phút
Epoch [ 90/100] | Train Loss (MSE): 0.049532 | Val Loss (MSE): 0.470734 | Thời gian: 14.21 phút
Epoch [100/100] 

In [5]:
# ===== CELL 5: ĐÁNH GIÁ TRÊN TẬP TEST =====
import joblib # Import joblib để nạp file lưu bộ giải chuẩn hóa Scaler[cite: 3]

# 1. Nạp lại bộ Scaler của Target đã được fit ở bước tiền xử lý[cite: 3]
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# 2. Nạp bản trọng số mô hình tốt nhất đã lưu trên Google Drive[cite: 3, 4]
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth'))
model.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trọng số tối ưu
model.eval()                                           # Chuyển mô hình sang chế độ dự đoán

# 3. Dự đoán toàn bộ tập Test (Data5 - 30,000 mẫu)[cite: 3]
all_preds = []   # Mảng lưu kết quả dự đoán đã chuẩn hóa
all_targets = [] # Mảng lưu nhãn thực tế đã chuẩn hóa

with torch.no_grad(): # Tắt cơ chế tính gradient
    for inputs, targets in test_loader:
        inputs = inputs.to(device)                     # Đẩy dữ liệu Batch lên GPU
        outputs = model(inputs)                       # Mạng MLP dự đoán
        all_preds.append(outputs.cpu().numpy())       # Chuyển kết quả dự đoán từ GPU về CPU dưới dạng NumPy Array
        all_targets.append(targets.numpy())           # Lưu nhãn thực tế dưới dạng NumPy Array

# Ghép danh sách các Batch thành mảng 2D hoàn chỉnh
all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# 4. Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực tế (Ohm cho Rs, Henry cho Ld và Lq)[cite: 3]
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# 5. Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
# Công thức: MAPE = Mean( | (Thực_tế - Dự_đoán) / Thực_tế | ) * 100%
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình: {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình: {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình: {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===
Sai số Rs trung bình: 7.59%
Sai số Ld trung bình: 0.27%
Sai số Lq trung bình: 0.34%
